In [ ]:
import osmnx as ox
import networkx as nx
import folium
import random
import geopandas as gpd

In [36]:

# 1. 그래프 G를 노드(점)와 엣지(선) 데이터프레임으로 분리
# 이 한 줄이면 G 안의 모든 복잡한 데이터가 표(Table) 형태로 예쁘게 정리됩니다.
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)

# 2. (선택) 저장할 때 에러를 방지하기 위해 리스트나 딕셔너리 형태의 컬럼을 문자열로 변환
# OSMnx 데이터 중 일부(예: osmid, lanes)는 리스트 형태라 DB에 바로 안 들어갈 수 있습니다.
for col in gdf_edges.columns:
    if any(isinstance(val, list) for val in gdf_edges[col]):
        gdf_edges[col] = gdf_edges[col].astype(str)

# 3. GeoPackage 로컬 DB 파일로 저장
# 하나의 gpkg 파일 안에 'nodes' 테이블과 'edges' 테이블을 각각 저장합니다.
gpkg_path = "busan_haeundae_network.gpkg"
gdf_nodes.to_file(gpkg_path, layer='nodes', driver="GPKG")
gdf_edges.to_file(gpkg_path, layer='edges', driver="GPKG")

print("GeoPackage DB화 완료!")

GeoPackage DB화 완료!


In [49]:
for u, v, key, data in G.edges(keys=True, data=True):
    print(f"Edge from {u} to {v} (key={key}): {data}")

Edge from 414663421 to 4823182628 (key=0): {'osmid': 37402559, 'highway': 'secondary', 'name': '재반로', 'oneway': False, 'reversed': False, 'length': np.float64(49.588798084403024), 'geometry': <LINESTRING (129.126 35.179, 129.126 35.179, 129.126 35.179)>}
Edge from 414663421 to 5265346729 (key=0): {'osmid': 37402559, 'highway': 'secondary', 'name': '재반로', 'oneway': False, 'reversed': True, 'length': np.float64(9.334189053945064), 'geometry': <LINESTRING (129.126 35.179, 129.126 35.179, 129.126 35.179)>}
Edge from 414663421 to 4823182634 (key=0): {'osmid': 1494558475, 'highway': 'secondary', 'name': '해운대로', 'oneway': True, 'ref': '31', 'reversed': False, 'length': np.float64(46.680462983931605), 'geometry': <LINESTRING (129.126 35.179, 129.126 35.179, 129.126 35.179)>}
Edge from 414663497 to 5265397093 (key=0): {'osmid': 1494558483, 'highway': 'secondary', 'name': '해운대로161번길', 'oneway': False, 'reversed': True, 'length': np.float64(10.879390177909853)}
Edge from 414663497 to 1791962459 (

### 0. 도로망 데이터 가져오기

#### 0.1. place 입력

In [ ]:
# 1. 특정 지역의 도로망 데이터 가져오기 (예: 부산 해운대구)
place_name = "Haeundae-gu, Busan, South Korea"

In [ ]:
try:
    G = ox.graph_from_place(place_name, network_type="walk")
    print(f"다운로드 완료: 노드 {len(G.nodes)}개, 엣지(도로) {len(G.edges)}개")
except Exception as e:
    print(f"데이터 다운로드 실패. 다른 지역명을 시도하거나 네트워크 상태를 확인하세요: {e}")
    exit()


: 

In [ ]:

# ---------------------------------------------------------
# 2. 고도 데이터 맵핑 (POC를 위한 임의 세팅)
# ---------------------------------------------------------
# 고도 범위: 0m ~ 10m (해안가 가정)
for node, data in G.nodes(data=True):
    # 테스트용: 무작위 고도 부여
    data['elevation'] = random.uniform(0.0, 10.0) 

# ---------------------------------------------------------
# 3. 해수면 상승 시뮬레이션 적용 (가중치 조작)
# ---------------------------------------------------------
# [수정 포인트] 시뮬레이션을 위해 상승폭을 조정합니다. 
# 고도가 0~10m인데 상승폭이 120m이면 모두 침수되어 경로를 찾을 수 없습니다.
sea_level_rise = 5.0  # 해수면 5.0m 상승 가정

for u, v, key, data in G.edges(keys=True, data=True):
    # 도로 양 끝 노드의 고도를 가져와 평균 고도 계산
    try:
        u_elev = G.nodes[u]['elevation']
        v_elev = G.nodes[v]['elevation']
        avg_elev = (u_elev + v_elev) / 2
        
        # 평균 고도가 해수면보다 낮으면 침수 처리
        if avg_elev < sea_level_rise:
            # 가중치를 무한대로 설정하여 길찾기에서 배제되도록 함
            data['simulation_cost'] = float('inf') 
            data['is_flooded'] = True
        else:
            # 침수되지 않은 도로는 원래의 거리(length)를 가중치로 사용
            # length 속성이 없는 경우를 대비해 기본값 1 부여
            data['simulation_cost'] = data.get('length', 1.0)
            data['is_flooded'] = False
    except KeyError:
        # 드물게 노드 데이터가 없는 경우 안전하게 처리
        data['simulation_cost'] = data.get('length', 1.0)
        data['is_flooded'] = False

# ---------------------------------------------------------
# 4. 길찾기 수행 (NetworkX 최단 경로 알고리즘)
# ---------------------------------------------------------
# 그래프 내에서 유효한 출발지, 도착지 노드 추출
nodes = list(G.nodes())
# 데이터에 따라 인덱스가 유효하지 않을 수 있으므로 안전하게 추출
if len(nodes) < 200:
    print("그래프 노드 수가 너무 적어 실험에 부적합합니다.")
    exit()

orig_node = nodes[50]   # 인덱스 조정
dest_node = nodes[-50]

print(f"출발 노드: {orig_node}, 도착 노드: {dest_node} 경로 탐색 중...")

route = None
try:
    # 침수 비용(simulation_cost)을 고려하여 길찾기 수행
    # dijkstra나 astar 등 사용 가능. 여기선 shortest_path 사용.
    route = nx.shortest_path(G, orig_node, dest_node, weight='simulation_cost')
    print(f"경로 탐색 성공! 경로를 구성하는 노드 수: {len(route)}개")
except nx.NetworkXNoPath:
    print("해수면 상승으로 인해 목적지로 가는 모든 경로가 침수되어 고립되었습니다! (우회 경로 없음)")
except nx.NodeNotFound as e:
    print(f"노드를 찾을 수 없습니다: {e}")

# ---------------------------------------------------------
# 5. 결과 시각화 (Folium 맵 생성)
# ---------------------------------------------------------
map_center = [G.nodes[orig_node]['y'], G.nodes[orig_node]['x']]
m = folium.Map(location=map_center, zoom_start=14, tiles="OpenStreetMap") # 배경을 회색조로 하면 물이 더 잘보임

# [새로 추가된 핵심 로직: 점들을 겹쳐서 '면'으로 만들기]
# 침수된 노드(교차로)를 찾아 그 주변에 넓은 반경의 물(Circle)을 채웁니다.
for node, data in G.nodes(data=True):
    if 'elevation' in data and data['elevation'] < sea_level_rise:
        folium.Circle(
            location=[data['y'], data['x']],
            radius=10,             # 반경 150m (수치를 키우면 면이 더 크게 합쳐짐)
            stroke=False,           # 테두리 선 없애기 (선이 없어야 자연스럽게 겹침)
            fill=True,
            fill_color='RoyalBlue', # 물 색상
            fill_opacity=0.4        # 반투명하게 설정하여 아래 지도가 보이도록
        ).add_to(m)

# 5-1. (선택) 끊긴 도로는 빨간색 점선으로 희미하게 표시 (물 아래 잠긴 느낌)
for u, v, key, data in G.edges(keys=True, data=True):
    if data.get('is_flooded'):
        u_coord = (G.nodes[u]['y'], G.nodes[u]['x'])
        v_coord = (G.nodes[v]['y'], G.nodes[v]['x'])
        folium.PolyLine([u_coord, v_coord], color="red", weight=2, opacity=0.3, dash_array='5, 5').add_to(m)

# 5-2. 안전한 우회 경로는 눈에 띄게 표시
if route:
    route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in route]
    folium.PolyLine(route_coords, color="LimeGreen", weight=6, opacity=1).add_to(m)
    
    folium.Marker(route_coords[0], popup="출발", icon=folium.Icon(color='green')).add_to(m)
    folium.Marker(route_coords[-1], popup="도착", icon=folium.Icon(color='blue')).add_to(m)

m.save('sea_level_surface_simulation.html')
print("면 형태 침수 지도 저장 완료!")

In [46]:
import osmnx as ox
import rasterio

In [ ]:
# 1. 대상 지역 도로망 다운로드
G = ox.graph_from_place("Haeundae-gu, Busan, South Korea", network_type="drive")

# 2. 실제 DEM 데이터(.tif) 로드 및 노드 고도 Join
dem_file_path = "해운대_고도데이터.tif" # (국토 정보플랫폼 등에서 다운로드한 파일)

with rasterio.open(dem_file_path) as dem_src:
    # 2-1. NetworkX 노드들의 (경도, 위도) 좌표만 리스트로 추출
    # 주의: DEM 파일의 좌표계(CRS)와 OSM의 좌표계(EPSG:4326)가 일치해야 합니다!
    coords = [(data['x'], data['y']) for node, data in G.nodes(data=True)]
    
    # 2-2. rasterio.sample을 통해 좌표에 해당하는 픽셀(고도) 값들을 한 번에 추출
    # sample()은 제너레이터를 반환하므로 리스트로 변환
    sampled_elevations = list(dem_src.sample(coords))
    
    # 2-3. 추출한 고도값을 다시 NetworkX 노드에 업데이트
    for i, (node, data) in enumerate(G.nodes(data=True)):
        # 추출된 값은 배열 형태이므로 첫 번째 값([0])을 가져옴
        elevation_value = sampled_elevations[i][0]
        
        # NoData 값(보통 -9999 등) 처리
        if elevation_value < -100: 
            data['elevation'] = 0.0 # 바다 등으로 간주
        else:
            data['elevation'] = float(elevation_value)

print("실제 고도 데이터 Join 완료!")

RasterioIOError: 해운대_고도데이터.tif: No such file or directory

In [ ]:
import folium
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
import base64

# (이전 단계에서 G를 이용한 길찾기 route 생성이 완료되었다고 가정)
sea_level_rise = 5.0

# 1. Folium 맵 초기화
map_center = [G.nodes[orig_node]['y'], G.nodes[orig_node]['x']]
m = folium.Map(location=map_center, zoom_start=14, tiles="CartoDB positron")

# ========================================================
# 2. [핵심] DEM 픽셀 데이터를 침수 레이어(이미지)로 변환
# ========================================================
with rasterio.open(dem_file_path) as dem_src:
    # DEM의 Z(고도) 배열 읽어오기
    dem_array = dem_src.read(1)
    
    # 지도의 경계(Bounds) 추출: [ [남단 위도, 서단 경도], [북단 위도, 동단 경도] ]
    # Folium ImageOverlay 배치를 위해 필수적임
    bounds = [
        [dem_src.bounds.bottom, dem_src.bounds.left],
        [dem_src.bounds.top, dem_src.bounds.right]
    ]

# 투명한 빈 이미지 캔버스(RGBA) 생성 (DEM 배열과 동일한 크기)
img_data = np.zeros((dem_array.shape[0], dem_array.shape[1], 4), dtype=np.uint8)

# 해수면보다 낮으면서, 유효한 데이터(NoData -9999가 아닌 곳)를 찾아 마스킹
flooded_mask = (dem_array < sea_level_rise) & (dem_array > -100)

# 침수된 픽셀만 반투명한 물 색상(RoyalBlue)으로 채우기
img_data[flooded_mask] = [65, 105, 225, 150] # [R, G, B, Alpha]

# 배열을 PNG 이미지로 변환 (메모리 상에서 처리)
image_file = BytesIO()
plt.imsave(image_file, img_data, format='png')
image_file.seek(0)
img_base64 = base64.b64encode(image_file.read()).decode('utf-8')
img_url = f"data:image/png;base64,{img_base64}"

# 3. Folium 지도 위에 생성한 침수 레이어(면) 오버레이
folium.raster_layers.ImageOverlay(
    image=img_url,
    bounds=bounds,
    opacity=0.8,
    name=f'해수면 {sea_level_rise}m 상승 침수 면적'
).add_to(m)

# ========================================================
# 4. 도로망(선) 및 우회 경로 시각화 추가
# ========================================================
# 침수 레이어를 깔았으므로, 끊긴 도로는 얇은 회색 또는 빨간색 점선으로 깔끔하게 표시
for u, v, key, data in G.edges(keys=True, data=True):
    if data.get('is_flooded'):
        u_coord = (G.nodes[u]['y'], G.nodes[u]['x'])
        v_coord = (G.nodes[v]['y'], G.nodes[v]['x'])
        folium.PolyLine([u_coord, v_coord], color="red", weight=1, opacity=0.4, dash_array='4').add_to(m)

# 안전한 길찾기 경로는 두껍고 선명하게 표시
if route:
    route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in route]
    folium.PolyLine(route_coords, color="LimeGreen", weight=5, opacity=1.0).add_to(m)

folium.LayerControl().add_to(m)
m.save('perfect_architecture_simulation.html')